In [1]:
import pandas as pd
import seaborn as sns
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import copy
from tqdm import tqdm
import numpy as np
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import torch
from torch.utils.data import DataLoader, SequentialSampler, TensorDataset, Subset
from torch.nn import CrossEntropyLoss, MSELoss
from functools import partial
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report
from sklearn.datasets import make_classification
import gc
import tempfile
import os

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(device)
else:
    print("MPS device not found.")

In [3]:
model = AutoModelForSequenceClassification.from_pretrained("fabriceyhc/bert-base-uncased-yahoo_answers_topics")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [4]:
class TestDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.sentences = df.iloc[:, 1:].apply(lambda x: ' '.join(x.dropna().astype(str)), axis=1).values
        self.labels = df.iloc[:, 0].values - 1  # Label을 0부터 시작하도록 조정
        self.tokenizer = tokenizer
    def __len__(self):
        return len(self.sentences)
    def __getitem__(self, idx):
        sentence = self.sentences[idx]
        inputs = self.tokenizer(sentence, truncation=True, max_length=512, padding='max_length', return_tensors="pt")
        label = torch.tensor(self.labels[idx])
        return inputs, label

In [5]:
# validation 데이터 가져오기
validation_data = "./validation.csv"
validation_df = pd.read_csv(validation_data, header=None, names=["class","question_title","question_body","answer"])
validation_dataset = TestDataset(validation_df, tokenizer)
validation_loader = DataLoader(validation_dataset, batch_size=32, shuffle=False)

# test 데이터 가져오기
test_data = "./test2.csv"
test_df = pd.read_csv(test_data, header=None, names=["class","question_title","question_body","answer"])
test_dataset = TestDataset(test_df, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [6]:
def eval_model(model, test_loader, device):
    preds = []
    true_labels = []
    embedding = []
    for batch in tqdm(test_loader, desc = "Evaluating model"):
        inputs, labels = batch
        inputs = {k: v.squeeze(1).to(device) for k, v in inputs.items()}
        labels = labels.to(device)

        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
        prediction = outputs.logits.argmax(dim=-1)

        preds.extend(prediction.tolist())
        true_labels.extend(labels.tolist())
        cls_embeddings = outputs.hidden_states[-1][:, 0, :].cpu().numpy()
        embedding.extend(cls_embeddings)
        report = classification_report(true_labels, preds)
    return preds, true_labels, report, np.array(embedding)

In [7]:
def extract_embeddings(outputs):
    return outputs.hidden_states[0], outputs.hidden_states[-1]

In [8]:
def select_true_example(model, num_labels, data_frame, data_loader, tokenizer, device):
    class_dfs = [[] for _ in range(num_labels)]
    temp_input_files = [tempfile.NamedTemporaryFile(delete=False) for _ in range(num_labels)]
    temp_output_files = [tempfile.NamedTemporaryFile(delete=False) for _ in range(num_labels)]
    
    for batch_start_idx, batch in enumerate(tqdm(data_loader, desc="Evaluating model"), start=0):
        inputs, labels = batch
        inputs = {k: v.squeeze(1).to(device) for k, v in inputs.items()}
        labels = labels.to(device)
        
        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
            input_embeddings, output_embeddings = extract_embeddings(outputs)
            
        predictions = outputs.logits.argmax(dim=-1)
        
        for i in range(len(labels)):
            if predictions[i] == labels[i]:
                original_index = batch_start_idx * data_loader.batch_size + i
                logit_value = outputs.logits[i, predictions[i]].item()
                original_data = data_frame.iloc[original_index].tolist()
                
                # Save embeddings to temporary files
                np.save(temp_input_files[labels[i].item()], input_embeddings[i].cpu().numpy())
                np.save(temp_output_files[labels[i].item()], output_embeddings[i].cpu().numpy())
                
                class_dfs[labels[i].item()].append((logit_value, original_data))
        
        # 메모리 해제
        del inputs, labels, outputs, input_embeddings, output_embeddings
        torch.cuda.empty_cache()
        gc.collect()

    sorted_input_embeddings = [[] for _ in range(num_labels)]
    sorted_output_embeddings = [[] for _ in range(num_labels)]
    
    for i in range(num_labels):
        # logit 값에 따라 정렬
        class_dfs[i].sort(key=lambda x: x[0], reverse=True)
        
        # 임시 파일에서 임베딩을 다시 읽어오면서 정렬된 순서대로 저장
        temp_input_files[i].seek(0)
        temp_output_files[i].seek(0)
        
        sorted_input_embeddings[i] = [np.load(temp_input_files[i]) for _ in class_dfs[i]]
        sorted_output_embeddings[i] = [np.load(temp_output_files[i]) for _ in class_dfs[i]]
        
        # 리스트를 DataFrame으로 변환
        class_dfs[i] = pd.DataFrame([x[1] for x in class_dfs[i]])
        
        # 임시 파일 삭제
        os.unlink(temp_input_files[i].name)
        os.unlink(temp_output_files[i].name)
    
    return sorted_input_embeddings, sorted_output_embeddings

In [9]:
def extract_top_n_embeddings(extract_num, class_num, embeddings):
    extracted_embeddings = []
    for i in range(0, class_num):
        class_embeddings = embeddings[i][:extract_num]
        class_embeddings_np = np.array(class_embeddings)
        class_embeddings_tensor = torch.tensor(class_embeddings_np)
        extracted_embeddings.append(class_embeddings_tensor)
    return extracted_embeddings

In [10]:
def get_decimal_precision(value):
    str_value = str(value)
    if '.' in str_value:
        return len(str_value.split('.')[1])
    else:
        return 0

In [11]:
def perturb(embeddings, eps, grad):
    perturbed_embeddings = embeddings - eps*grad.sign()
    return perturbed_embeddings

In [12]:
def calculate_gradient(model, input_embeddings, target_class, device):
    input_embeddings = input_embeddings.clone().detach().requires_grad_(True).to(device)
    target = torch.tensor([target_class]).to(device)
    outputs = model(inputs_embeds=input_embeddings, labels=target)
    init_pred = outputs.logits.argmax(dim=-1)
    loss = outputs.loss
    model.zero_grad()
    loss.backward()
    data_grad = input_embeddings.grad
    return loss, data_grad

In [13]:
def fgsm_attack(model, target_class, input_embeddings, start_epsilon, epsilon_step, max_epsilon, device):
    input_embeddings = input_embeddings.to(device)
    digits = get_decimal_precision(epsilon_step)
    eps = start_epsilon
    loss, data_grad = calculate_gradient(model, input_embeddings, target_class, device)
    while eps <= max_epsilon:
        # print(f"epsilon : {eps}")
        perturbed_embeddings = perturb(input_embeddings, eps, data_grad)
        adv_outputs = model(inputs_embeds = perturbed_embeddings)
        adv_pred = adv_outputs.logits.argmax(dim=-1)
        # print(adv_pred)
        if adv_pred.item() == target_class:
            # print(f"{eps} adv attack success!")
            # print()
            break
        else:
            eps += epsilon_step
            eps = round(eps, digits)
            # print()
    return eps, perturbed_embeddings

In [14]:
def calculate_all_epsilon(model, class_num, top_n_logit_example, step_epsilon=0.01, max_epsilon=10.0):
    epsilon_list = []
    max_eps = max_epsilon
    
    for i in range(class_num):
        print(f"Calculating epsilon for class {i}")
        temp = []
        for j in range(class_num):
            if i == j:
                temp.append(np.inf)
                continue
            
            # Calculate epsilon for each pair (i, j) and free memory after each calculation
            with torch.no_grad():
                epsilon, perturbed = fgsm_attack(model, j, top_n_logit_example[i], 0.00, step_epsilon, max_eps, device)
            if epsilon >= max_eps:
                temp.append(np.inf)
            else:
                temp.append(epsilon)
                
            # Free memory
            del perturbed
            torch.cuda.empty_cache()
            gc.collect()
        
        epsilon_list.append(temp)
    return epsilon_list

In [15]:
def generate_example(model, start_epsilon, end_epsilon, source_class, target_class, input_embeddings, device, per_attack_example_num, min_step_eps=1e-5):
    example_list = []
    example_label = []
    source_embedding = input_embeddings[source_class].to(device)
    # Gradient 계산
    loss, data_grad = calculate_gradient(model, source_embedding, target_class, device)
    # step_eps 계산 및 최소값 설정
    step_eps = (end_epsilon - start_epsilon) / per_attack_example_num
    step_eps = max(step_eps, min_step_eps)  # 최소값 설정
    iter_eps = start_epsilon
    iter_num = 0
    while iter_eps < end_epsilon and iter_num < per_attack_example_num:
        # Perturbation 적용
        generated_embeddings = perturb(source_embedding, iter_eps, data_grad)
        example_list.append(generated_embeddings.cpu())  # CPU로 이동
        example_label.append(source_class)
        iter_eps += step_eps  # iter_eps 증가
        iter_num += 1
        del generated_embeddings
        torch.cuda.empty_cache()  # GPU 메모리 캐시 비우기
    del source_embedding, loss, data_grad
    torch.cuda.empty_cache()  # 캐시 비우기
    return example_label, example_list

In [16]:
def make_example(model, data_frame, data_loader, tokenizer, example_num, emb_num, class_num, true_ratio, device, step_epsilon=0.01, max_epsilon=10.0):
    positive_example_list = []
    positive_example_label = []
    negative_example_list = []
    negative_example_label = []
    # Positive, Negative 예제 개수 계산
    true_ratio *= 0.01
    positive_num = int(round(example_num * true_ratio))
    negative_num = example_num - positive_num
    print("positive num : ", positive_num)
    print("negative num : ", negative_num)
    
    # 입력된 embedding 중 상위 N개를 추출
    input_embeddings, output_embeddings = select_true_example(model, class_num, data_frame, data_loader, tokenizer, device)
    print("done0")
    extract_top_n_emb = extract_top_n_embeddings(emb_num, class_num, input_embeddings)
    print("done1")
    epsilon_list = calculate_all_epsilon(model, class_num, extract_top_n_emb, step_epsilon, max_epsilon)
    print(epsilon_list)

    per_class_positive_example_num = int(round(positive_num / class_num))
    per_class_negative_example_num = int(round(negative_num / class_num))
    
    print("per_class_positive_example_num : ", per_class_positive_example_num)
    print("per_class_negative_example_num : ", per_class_negative_example_num)
    
    for source_class in range(class_num):
        inf_num = sum(math.isinf(item) for item in epsilon_list[source_class])
        per_target_positive_example_num = int(round(per_class_positive_example_num / (class_num - inf_num)))
        per_target_negative_example_num = int(round(per_class_negative_example_num / (class_num - inf_num)))
        
        # print(per_target_positive_example_num)
        # print(per_target_negative_example_num)
        for target_class in range(class_num):
            epsilon = epsilon_list[source_class][target_class]
            if math.isinf(epsilon):
                continue
            else:
                # Positive 예제 생성
                pos_label, pos_examples = generate_example(
                    model,
                    start_epsilon=0.00,
                    end_epsilon=epsilon - step_epsilon,
                    source_class=source_class,
                    target_class=target_class,
                    input_embeddings=extract_top_n_emb,
                    device=device,
                    per_attack_example_num=per_target_positive_example_num
                )
                positive_example_label.extend(pos_label)
                positive_example_list.extend(pos_examples)
                # Negative 예제 생성
                neg_label, neg_examples = generate_example(
                    model,
                    start_epsilon=epsilon,
                    end_epsilon=2 * epsilon - step_epsilon,
                    source_class=source_class,
                    target_class=target_class,
                    input_embeddings=extract_top_n_emb,
                    device=device,
                    per_attack_example_num=per_target_negative_example_num
                )
                negative_example_label.extend(neg_label)
                negative_example_list.extend(neg_examples)
    return positive_example_label, positive_example_list, negative_example_label, negative_example_list

In [26]:
positive_example_label, positive_example_list, negative_example_label, negative_example_list = make_example(model, data_frame = validation_df, data_loader = validation_loader, tokenizer=tokenizer, example_num=3000, emb_num=1, class_num=10, true_ratio=90, device=device, step_epsilon=0.01, max_epsilon=10.0)

positive num :  2700
negative num :  300


Evaluating model: 100%|██████████| 938/938 [04:13<00:00,  3.69it/s]


done0
done1
Calculating epsilon for class 0


RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [ ]:
def select_true_example(model, num_labels, data_frame, data_loader, tokenizer, device):
    correct_predictions = [[] for _ in range(num_labels)]
    sorted_input_embeddings = []
    sorted_output_embeddings = []
    class_dfs = []
    batch_start_idx = 0
    
    for batch in tqdm(data_loader, desc="Evaluating model"):
        inputs, labels = batch
        inputs = {k: v.squeeze(1).to(device) for k, v in inputs.items()}
        labels = labels.to(device)
        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
            input_embeddings, output_embeddings = extract_embeddings(outputs)
        predictions = outputs.logits.argmax(dim=-1)
        for i in range(len(labels)):
            if predictions[i] == labels[i]:
                original_index = batch_start_idx + i
                correct_predictions[labels[i].item()].append((outputs.logits[i, predictions[i]].item(), data_frame.iloc[original_index].tolist(), input_embeddings[i].cpu().numpy(), output_embeddings[i].cpu().numpy()))
        batch_start_idx += len(labels)
        
    for i in range(num_labels):
        # logit 값에 따라 정렬
        correct_predictions[i].sort(key=lambda x: x[0], reverse=True)
        # 정렬된 데이터를 분리하여 저장
        sorted_class_data = [pred[1] for pred in correct_predictions[i]]
        sorted_input_embeds = [pred[2] for pred in correct_predictions[i]]
        sorted_output_embeds = [pred[3] for pred in correct_predictions[i]]
        # 정렬된 데이터를 DataFrame으로 변환
        class_df = pd.DataFrame(sorted_class_data)
        class_dfs.append(class_df)
        sorted_input_embeddings.append(sorted_input_embeds)
        sorted_output_embeddings.append(sorted_output_embeds)
    return class_dfs, sorted_input_embeddings, sorted_output_embeddings, correct_predictions

In [29]:
def select_true_example(model, num_labels, data_frame, data_loader, tokenizer, device):
    class_dfs = [[] for _ in range(num_labels)]
    sorted_input_embeddings = [[] for _ in range(num_labels)]
    sorted_output_embeddings = [[] for _ in range(num_labels)]
    
    for batch_start_idx, batch in enumerate(tqdm(data_loader, desc="Evaluating model"), start=0):
        inputs, labels = batch
        inputs = {k: v.squeeze(1).to(device) for k, v in inputs.items()}
        labels = labels.to(device)
        
        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
            input_embeddings, output_embeddings = extract_embeddings(outputs)
            
        predictions = outputs.logits.argmax(dim=-1)
        
        for i in range(len(labels)):
            if predictions[i] == labels[i]:
                original_index = batch_start_idx * data_loader.batch_size + i
                logit_value = outputs.logits[i, predictions[i]].item()
                original_data = data_frame.iloc[original_index].tolist()
                input_embed = input_embeddings[i].cpu().numpy()
                output_embed = output_embeddings[i].cpu().numpy()
                
                class_dfs[labels[i].item()].append((logit_value, original_data, input_embed, output_embed))
        
        # 메모리 해제
        del inputs, labels, outputs, input_embeddings, output_embeddings
        torch.cuda.empty_cache()
        gc.collect()

    for i in range(num_labels):
        # logit 값에 따라 정렬
        class_dfs[i].sort(key=lambda x: x[0], reverse=True)
        
        # 정렬된 데이터를 각각의 리스트에 저장
        for logit, original_data, input_embed, output_embed in class_dfs[i]:
            sorted_input_embeddings[i].append(input_embed)
            sorted_output_embeddings[i].append(output_embed)
            # 필요에 따라 메모리 절약을 위해 데이터 프레임에 추가하기 전에 최소한의 데이터만 저장할 수 있습니다.
        
        # 리스트를 DataFrame으로 변환
        class_dfs[i] = pd.DataFrame([x[1] for x in class_dfs[i]])
    
    return class_dfs, sorted_input_embeddings, sorted_output_embeddings

In [7]:
# 모델 평가
predictions, true_labels, report, embeddings = eval_model(model, test_loader, device)

Evaluating model:   0%|          | 0/938 [00:00<?, ?it/s]/home/jieungkim/anaconda3/envs/DecomposeTransformer/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/jieungkim/anaconda3/envs/DecomposeTransformer/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/jieungkim/anaconda3/envs/DecomposeTransformer/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples

In [9]:
# 임의로 예측 클래스 이름 설정 (여기서는 10개의 클래스를 가정)
class_names = [f"Class {i}" for i in range(10)]

# t-SNE 적용
tsne = TSNE(n_components=2, random_state=42)
reduced_embeddings = tsne.fit_transform(embeddings)

# 클래스별로 색깔이 구분되도록 시각화
plt.figure(figsize=(12, 8))
scatter = plt.scatter(reduced_embeddings[:, 0], reduced_embeddings[:, 1], c=predictions, cmap='tab10', edgecolor='k', s=40)

# 범례 생성
handles, labels = scatter.legend_elements(prop="colors", alpha=0.6)
legend_labels = ["class 1", "class 2", "class 3", "class 4", "class 5", "class 6", "class 7", "class 8", "class 9", "class 10"]
legend = plt.legend(handles, legend_labels, title="Classes")

plt.title("t-SNE visualization of CLS token embeddings by predicted class")
plt.xlabel("Dimension 1")
plt.ylabel("Dimension 2")
plt.show()

NameError: name 'embeddings' is not defined

In [4]:
import pandas as pd
import seaborn as sns
from torch.utils.data import DataLoader, Dataset,  SequentialSampler, TensorDataset, Subset
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import copy
from tqdm import tqdm
import numpy as np
from sklearn.decomposition import PCA
import torch
from torch.nn import CrossEntropyLoss, MSELoss
from functools import partial
import torch.nn.functional as F
import matplotlib.pyplot as plt
import math
from sklearn.metrics import classification_report

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(device)
else:
    print("MPS device not found.")

model = AutoModelForSequenceClassification.from_pretrained("fabriceyhc/bert-base-uncased-yahoo_answers_topics")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model.to(device)

class TestDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.sentences = df.iloc[:, 1:].apply(lambda x: ' '.join(x.dropna().astype(str)), axis=1).values
        self.labels = df.iloc[:, 0].values - 1  # Label을 0부터 시작하도록 조정
        self.tokenizer = tokenizer
    def __len__(self):
        return len(self.sentences)
    def __getitem__(self, idx):
        sentence = self.sentences[idx]
        inputs = self.tokenizer(sentence, truncation=True, max_length=512, padding='max_length', return_tensors="pt")
        label = torch.tensor(self.labels[idx])
        return inputs, label

# validation 데이터 가져오기
validation_data = "./validation.csv"
validation_df = pd.read_csv(validation_data, header=None, names=["class","question_title","question_body","answer"])
validation_dataset = TestDataset(validation_df, tokenizer)
validation_loader = DataLoader(validation_dataset, batch_size=32, shuffle=False)
# test 데이터 가져오기
test_data = "./test2.csv"
test_df = pd.read_csv(test_data, header=None, names=["class","question_title","question_body","answer"])
test_dataset = TestDataset(test_df, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

def extract_embeddings(outputs):
    return outputs.hidden_states[0], outputs.hidden_states[-1]

def select_true_example(model, num_labels, data_frame, data_loader, tokenizer, device):
    correct_predictions = [[] for _ in range(num_labels)]
    sorted_input_embeddings = []
    sorted_output_embeddings = []
    class_dfs = []
    batch_start_idx = 0
    for batch in tqdm(data_loader, desc="Evaluating model"):
        inputs, labels = batch
        inputs = {k: v.squeeze(1).to(device) for k, v in inputs.items()}
        labels = labels.to(device)
        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
            input_embeddings, output_embeddings = extract_embeddings(outputs)
        predictions = outputs.logits.argmax(dim=-1)
        for i in range(len(labels)):
            if predictions[i] == labels[i]:
                # 올바른 인덱스를 사용하여 데이터프레임의 해당 행을 가져옴
                original_index = batch_start_idx + i
                correct_predictions[labels[i].item()].append((outputs.logits[i, predictions[i]].item(), data_frame.iloc[original_index].tolist(), input_embeddings[i].cpu().numpy(), output_embeddings[i].cpu().numpy()))
        batch_start_idx += len(labels)
    for i in range(num_labels):
        # logit 값에 따라 정렬
        correct_predictions[i].sort(key=lambda x: x[0], reverse=True)
        # 정렬된 데이터를 분리하여 저장
        sorted_class_data = [pred[1] for pred in correct_predictions[i]]
        sorted_input_embeds = [pred[2] for pred in correct_predictions[i]]
        sorted_output_embeds = [pred[3] for pred in correct_predictions[i]]
        # 정렬된 데이터를 DataFrame으로 변환
        class_df = pd.DataFrame(sorted_class_data)
        class_dfs.append(class_df)
        sorted_input_embeddings.append(sorted_input_embeds)
        sorted_output_embeddings.append(sorted_output_embeds)
    return sorted_input_embeddings, sorted_output_embeddings

def extract_top_n_embeddings(extract_num, class_num, embeddings):
    extracted_embeddings = []
    for i in range(0, class_num):
        class_embeddings = embeddings[i][:extract_num]
        class_embeddings_np = np.array(class_embeddings)
        class_embeddings_tensor = torch.tensor(class_embeddings_np)
        extracted_embeddings.append(class_embeddings_tensor)
    return extracted_embeddings

def get_decimal_precision(value):
    str_value = str(value)
    if '.' in str_value:
        return len(str_value.split('.')[1])
    else:
        return 0

def perturb(embeddings, eps, grad):
    perturbed_embeddings = embeddings - eps*grad.sign()
    return perturbed_embeddings

def calculate_gradient(model, input_embeddings, target_class, device):
    input_embeddings = input_embeddings.clone().detach().requires_grad_(True).to(device)
    target = torch.tensor([target_class]).to(device)
    outputs = model(inputs_embeds=input_embeddings, labels=target)
    init_pred = outputs.logits.argmax(dim=-1)
    loss = outputs.loss
    model.zero_grad()
    loss.backward()
    data_grad = input_embeddings.grad
    return loss, data_grad

def fgsm_attack(model, target_class, input_embeddings, start_epsilon, epsilon_step, max_epsilon, device):
    input_embeddings = input_embeddings.to(device)
    digits = get_decimal_precision(epsilon_step)
    eps = start_epsilon
    loss, data_grad = calculate_gradient(model, input_embeddings, target_class, device)
    while eps <= max_epsilon:
        # print(f"epsilon : {eps}")
        perturbed_embeddings = perturb(input_embeddings, eps, data_grad)
        adv_outputs = model(inputs_embeds = perturbed_embeddings)
        adv_pred = adv_outputs.logits.argmax(dim=-1)
        # print(adv_pred)
        if adv_pred.item() == target_class:
            # print(f"{eps} adv attack success!")
            # print()
            break
        else:
            eps += epsilon_step
            eps = round(eps, digits)
            # print()
    return eps, perturbed_embeddings

def calculate_all_epsilon(model, class_num, top_n_logit_example, step_epsilon = 0.01, max_epsilon = 10.0):
    epsilon_list = []
    max_eps = max_epsilon
    for i in range (0, 10):
        print(f"class {i}")
        temp = []
        for j in range(0, 10):
            if i == j:
                temp.append(math.inf)
                continue
            epsilon, perterbed = fgsm_attack(model, j, top_n_logit_example[i], 0.00, step_epsilon, max_eps, device)
            if epsilon >= max_eps:
                temp.append(math.inf)
            else:
                temp.append(epsilon)
        epsilon_list.append(temp)
    return epsilon_list

def make_example(model, data_frame, data_loader, tokenizer, example_num, emb_num, class_num, true_ratio, device, step_epsilon=0.01, max_epsilon=10.0):
    positive_example_list = []
    positive_example_label = []
    negative_example_list = []
    negative_example_label = []
    # Positive, Negative 예제 개수 계산
    true_ratio *= 0.01
    positive_num = int(example_num * true_ratio)
    negative_num = example_num - positive_num
    print("positive num : ", positive_num)
    print("negative num : ", negative_num)
    # 입력된 embedding 중 상위 N개를 추출
    c_df, input_embeddings, output_embeddings, c_pd = select_true_example(model, class_num, data_frame, data_loader, tokenizer, device)
    extract_top_n_emb = extract_top_n_embeddings(emb_num, class_num, input_embeddings)
    epsilon_list = calculate_all_epsilon(model, class_num, extract_top_n_emb, step_epsilon, max_epsilon)
    print(epsilon_list)
    per_class_positive_example_num = int(positive_num / class_num)
    per_class_negative_example_num = int(negative_num / class_num)
    print("per_class_positive_example_num : ", per_class_positive_example_num)
    print("per_class_negative_example_num : ", per_class_negative_example_num)
    for source_class in range(class_num):
        inf_num = sum(math.isinf(item) for item in epsilon_list[source_class])
        per_target_positive_example_num = int(per_class_positive_example_num / (class_num - inf_num))
        per_target_negative_example_num = int(per_class_negative_example_num / (class_num - inf_num))
        # print(per_target_positive_example_num)
        # print(per_target_negative_example_num)
        for target_class in range(class_num):
            epsilon = epsilon_list[source_class][target_class]
            if math.isinf(epsilon):
                continue
            else:
                # Positive 예제 생성
                pos_label, pos_examples = generate_example(
                    model,
                    start_epsilon=0.00,
                    end_epsilon=epsilon - step_epsilon,
                    source_class=source_class,
                    target_class=target_class,
                    input_embeddings=extract_top_n_emb,
                    device=device,
                    per_attack_example_num=per_target_positive_example_num
                )
                positive_example_label.extend(pos_label)
                positive_example_list.extend(pos_examples)
                # Negative 예제 생성
                neg_label, neg_examples = generate_example(
                    model,
                    start_epsilon=epsilon,
                    end_epsilon=2 * epsilon - step_epsilon,
                    source_class=source_class,
                    target_class=target_class,
                    input_embeddings=extract_top_n_emb,
                    device=device,
                    per_attack_example_num=per_target_negative_example_num
                )
                negative_example_label.extend(neg_label)
                negative_example_list.extend(neg_examples)
    return positive_example_label, positive_example_list, negative_example_label, negative_example_list

def generate_example(model, start_epsilon, end_epsilon, source_class, target_class, input_embeddings, device, per_attack_example_num, min_step_eps=1e-5):
    example_list = []
    example_label = []
    source_embedding = input_embeddings[source_class].to(device)
    # Gradient 계산
    loss, data_grad = calculate_gradient(model, source_embedding, target_class, device)
    # step_eps 계산 및 최소값 설정
    step_eps = (end_epsilon - start_epsilon) / per_attack_example_num
    step_eps = max(step_eps, min_step_eps)  # 최소값 설정
    iter_eps = start_epsilon
    iter_num = 0
    while iter_eps < end_epsilon and iter_num < per_attack_example_num:
        # Perturbation 적용
        generated_embeddings = perturb(source_embedding, iter_eps, data_grad)
        example_list.append(generated_embeddings.cpu())  # CPU로 이동
        example_label.append(source_class)
        iter_eps += step_eps  # iter_eps 증가
        iter_num += 1
        del generated_embeddings
        torch.cuda.empty_cache()  # GPU 메모리 캐시 비우기
    del source_embedding, loss, data_grad
    torch.cuda.empty_cache()  # 캐시 비우기
    return example_label, example_list

cuda


In [5]:
input_embeddings, output_embeddings = select_true_example(model, 10, validation_df, validation_loader, tokenizer, device)
extract_top_n_emb = extract_top_n_embeddings(1, 10, input_embeddings)
epsilon_list = calculate_all_epsilon(model, 10, extract_top_n_emb, 0.01, 5.00)

Evaluating model: 100%|██████████| 938/938 [03:34<00:00,  4.38it/s]  


class 0
class 1
class 2
class 3
class 4
class 5
class 6
class 7
class 8
class 9


In [6]:
epsilon_list

[[inf, 0.6, 0.28, 0.24, inf, inf, inf, 0.26, 0.37, 0.12],
 [0.03, inf, 0.04, 0.06, 0.03, 0.74, inf, 0.51, inf, 0.04],
 [0.23, 0.07, inf, 0.33, inf, 3.53, inf, 0.16, 0.19, 0.08],
 [0.07, 0.04, 0.05, inf, 0.19, 0.21, 0.17, 0.53, 1.86, 0.08],
 [0.28, inf, inf, 0.1, inf, inf, inf, 0.03, inf, 0.01],
 [0.37, inf, 1.89, 0.5, inf, inf, inf, 0.03, inf, 0.5],
 [0.04, 0.06, 0.04, 0.16, 0.2, 0.16, inf, 0.32, 0.07, 0.04],
 [0.22, 0.01, 0.07, inf, 0.03, inf, inf, inf, 0.89, 0.54],
 [0.09, inf, 0.21, inf, inf, inf, inf, 0.02, inf, 0.69],
 [0.08, 0.03, 0.64, 0.06, inf, 0.56, inf, 0.21, 0.33, inf]]